In [0]:

%run ./00_transformations

In [0]:
import unittest
from datetime import date
from pyspark.sql import functions as F

CUST = "customer_id string, customer_name string, email string, phone string, segment string, country string, city string, state string, postal_code string, region string"
PROD = "product_id string, category string, sub_category string, product_name string, state string, price_per_product string"
ORD  = "row_id long, order_id string, order_date string, ship_date string, ship_mode string, customer_id string, product_id string, quantity long, price double, discount double, profit double"

def cust(cid, name, country="United States"):
    return (cid, name, None, None, None, country, None, None, None, None)

def order(rid, cid, pid, odate, profit):
    return (rid, f"O{rid}", odate, odate, "Standard Class", cid, pid, 1, 10.0, 0.0, profit)

class TestCleanCustomers(unittest.TestCase):
    def test_name_cleaned(self):
        df = spark.createDataFrame([cust("C1", "Gary567 Hansen"), cust("C2", "  _Mike Vitt 12313orini"),
                                    cust("C3", "Pete@#$ Takahito")], CUST)
        out = {r.customer_id: r.customer_name for r in clean_customers(df).collect()}
        self.assertEqual(out, {"C1": "Gary Hansen", "C2": "Mike Vitt Orini", "C3": "Pete Takahito"})

    def test_empty_and_null_name_become_null(self):
        df = spark.createDataFrame([cust("C1", None), cust("C2", "123 ###")], CUST)
        names = [r.customer_name for r in clean_customers(df).collect()]
        self.assertEqual(names, [None, None])

    def test_null_key_dropped_and_duplicates_removed(self):
        df = spark.createDataFrame([cust(None, "No Key"), cust("C1", "Ann Lee"), cust("C1", "Ann Lee")], CUST)
        self.assertEqual(clean_customers(df).count(), 1)

class TestCleanProducts(unittest.TestCase):
    def test_one_row_per_product_id_and_deterministic(self):
        rows = [("P1", "Furniture", "Chairs", "Chair A", "Texas", "10.5"),
                ("P1", "Furniture", "Chairs", "Chair B", "California", "12.0"),
                ("P2", "Technology", "Phones", "Phone", "Ohio", "99")]
        out1 = clean_products(spark.createDataFrame(rows, PROD))
        out2 = clean_products(spark.createDataFrame(rows[::-1], PROD))
        self.assertEqual(out1.count(), 2)
        self.assertEqual(sorted(out1.collect()), sorted(out2.collect()))

    def test_price_cast_to_double(self):
        df = spark.createDataFrame([("P1", "Furniture", "Chairs", "Chair", "Texas", "81.882")], PROD)
        self.assertEqual(dict(clean_products(df).dtypes)["price_per_product"], "double")

class TestEnrichedOrders(unittest.TestCase):
    def setUp(self):
        self.cust = clean_customers(spark.createDataFrame([cust("C1", "Ann Lee")], CUST))
        self.prod = clean_products(spark.createDataFrame(
            [("P1", "Furniture", "Chairs", "Chair", "Texas", "10")], PROD))

    def build(self, rows):
        return build_enriched_orders(spark.createDataFrame(rows, ORD), self.cust, self.prod)

    def test_profit_rounded_to_2_decimals(self):
        out = self.build([order(1, "C1", "P1", "21/8/2016", 10.12345), order(2, "C1", "P1", "21/8/2016", -3.456)])
        self.assertEqual(sorted(r.profit for r in out.collect()), [-3.46, 10.12])

    def test_date_parsed_without_leading_zeros_and_year(self):
        r = self.build([order(1, "C1", "P1", "6/10/2016", 1.0)]).first()
        self.assertEqual(r.order_date, date(2016, 10, 6))
        self.assertEqual(r.year, 2016)

    def test_customer_and_product_columns_joined(self):
        r = self.build([order(1, "C1", "P1", "1/1/2015", 1.0)]).first()
        self.assertEqual((r.customer_name, r.country, r.category, r.sub_category),
                         ("Ann Lee", "United States", "Furniture", "Chairs"))

    def test_unmatched_orders_are_kept_with_nulls(self):
        out = self.build([order(1, "C1", "P1", "1/1/2015", 5.0), order(2, "C9", "P9", "1/1/2015", 7.0)])
        self.assertEqual(out.count(), 2)
        bad = out.filter("row_id = 2").first()
        self.assertIsNone(bad.category); self.assertIsNone(bad.customer_name)

    def test_row_count_preserved_with_duplicate_products_in_source(self):
        dup = clean_products(spark.createDataFrame(
            [("P1", "Furniture", "Chairs", "A", "Texas", "1"), ("P1", "Furniture", "Chairs", "B", "Ohio", "2")], PROD))
        out = build_enriched_orders(spark.createDataFrame([order(1, "C1", "P1", "1/1/2015", 5.0)], ORD), self.cust, dup)
        self.assertEqual(out.count(), 1)

class TestAggregate(unittest.TestCase):
    def setUp(self):
        cu = clean_customers(spark.createDataFrame([cust("C1", "Ann Lee"), cust("C2", "Bob Ray")], CUST))
        pr = clean_products(spark.createDataFrame(
            [("P1", "Furniture", "Chairs", "Chair", "Texas", "10")], PROD))
        rows = [order(1, "C1", "P1", "1/1/2015", 10.0), order(2, "C1", "P1", "2/2/2015", 5.5),
                order(3, "C2", "P1", "3/3/2016", -2.0), order(4, "C2", "P9", "3/3/2016", 1.0)]
        self.enriched = build_enriched_orders(spark.createDataFrame(rows, ORD), cu, pr)

    def test_sums_by_group(self):
        agg = build_profit_aggregate(self.enriched)
        got = agg.filter("customer_id = 'C1' AND year = 2015").first()
        self.assertEqual(got.total_profit, 15.5)

    def test_grand_total_reconciles(self):
        agg = build_profit_aggregate(self.enriched)
        total_agg = agg.agg(F.round(F.sum("total_profit"), 2)).first()[0]
        total_src = self.enriched.agg(F.round(F.sum("profit"), 2)).first()[0]
        self.assertEqual(total_agg, total_src)

    def test_customers_with_same_name_not_merged(self):
        cu = clean_customers(spark.createDataFrame([cust("C1", "Sam Lee"), cust("C2", "Sam Lee")], CUST))
        pr = clean_products(spark.createDataFrame([("P1", "Furniture", "Chairs", "Chair", "Texas", "10")], PROD))
        e = build_enriched_orders(spark.createDataFrame(
            [order(1, "C1", "P1", "1/1/2015", 1.0), order(2, "C2", "P1", "1/1/2015", 2.0)], ORD), cu, pr)
        self.assertEqual(build_profit_aggregate(e).count(), 2)

    def test_empty_input_returns_empty(self):
        self.assertEqual(build_profit_aggregate(self.enriched.limit(0)).count(), 0)


In [0]:
classes = [TestCleanCustomers, TestCleanProducts, TestEnrichedOrders, TestAggregate]
suite = unittest.TestSuite(unittest.TestLoader().loadTestsFromTestCase(c) for c in classes)
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful(), "Some tests failed"
